## Example 1: Basic CMA-ES Optimization

In [ ]:
import cma
import numpy as np

# Example 1: Basic CMA-ES optimization of simple functions
print("Example 1: Testing basic CMA-ES optimization")

# Helper function for formatting numpy arrays
def format_array(arr, precision=6):
    """Format numpy array for printing"""
    return '[' + ', '.join(f'{x:.{precision}f}' for x in arr) + ']'

try:
    # Define simple test functions
    def sphere_function(x):
        """Simple sphere function: f(x) = sum(x^2)"""
        return sum(xi**2 for xi in x)
    
    def rosenbrock_function(x):
        """Rosenbrock function: classic optimization test function"""
        return sum(100 * (x[i+1] - x[i]**2)**2 + (1 - x[i])**2 for i in range(len(x)-1))
    
    print("Optimizing sphere function (minimum at origin):")
    
    # Initialize CMA-ES optimizer
    initial_guess = [2.0, 1.5, -1.0]  # Starting point away from optimum
    sigma0 = 1.0  # Initial standard deviation
    
    # Create and run optimizer
    es = cma.CMAEvolutionStrategy(initial_guess, sigma0, {'maxiter': 100})
    
    # Optimization loop
    iteration = 0
    while not es.stop() and iteration < 50:
        solutions = es.ask()
        fitness_values = [sphere_function(x) for x in solutions]
        es.tell(solutions, fitness_values)
        iteration += 1
        
        # Print progress every 10 iterations
        if iteration % 10 == 0:
            print(f"  Iteration {iteration}: Best fitness = {es.result.fbest:.6f}")
    
    # Get results
    result = es.result
    print(f"\nOptimization completed:")
    print(f"  Best solution: {format_array(result.xbest)}")
    print(f"  Best fitness: {result.fbest:.6f}")
    print(f"  Function evaluations: {result.evaluations}")
    print(f"  Iterations: {result.iterations}")
    
    print("✓ Basic CMA-ES optimization working")
    print("cma example 1 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    import traceback
    traceback.print_exc()
    raise
    
print("Example 1 finished")

## Example 2: Multi-Dimensional Optimization with Constraints

In [ ]:
import cma
import numpy as np

# Example 2: Multi-dimensional optimization with boundary constraints
print("Example 2: Testing multi-dimensional optimization with constraints")

# Helper function for formatting numpy arrays
def format_array(arr, precision=4):
    """Format numpy array for printing"""
    if len(arr) > 6:
        # Show first 3 and last 3 for long arrays
        formatted = ', '.join(f'{x:.{precision}f}' for x in arr[:3])
        formatted += ', ..., '
        formatted += ', '.join(f'{x:.{precision}f}' for x in arr[-3:])
        return '[' + formatted + ']'
    return '[' + ', '.join(f'{x:.{precision}f}' for x in arr) + ']'

try:
    def constrained_function(x):
        """Multi-modal function with local minima"""
        return sum((xi - 1)**2 + 0.1 * np.sin(10 * xi)**2 for xi in x)
    
    def penalty_function(x, bounds):
        """Add penalty for constraint violations"""
        penalty = 0
        for i, (lower, upper) in enumerate(bounds):
            if x[i] < lower:
                penalty += 1000 * (lower - x[i])**2
            elif x[i] > upper:
                penalty += 1000 * (x[i] - upper)**2
        return penalty
    
    print("Optimizing constrained multi-dimensional function:")
    
    # Problem setup
    dimensions = 5
    bounds = [(-2.0, 3.0)] * dimensions  # Box constraints
    initial_guess = [0.5] * dimensions
    sigma0 = 0.8
    
    # Configure optimizer with boundary handling
    options = {
        'bounds': [[-2] * dimensions, [3] * dimensions],
        'maxiter': 100,
        'tolfun': 1e-8,
        'verb_disp': 0  # Reduce output verbosity
    }
    
    es = cma.CMAEvolutionStrategy(initial_guess, sigma0, options)
    
    # Optimization with constraint handling
    iteration = 0
    best_fitness_history = []
    
    while not es.stop() and iteration < 80:
        solutions = es.ask()
        
        # Evaluate fitness with penalties
        fitness_values = []
        for x in solutions:
            base_fitness = constrained_function(x)
            penalty = penalty_function(x, bounds)
            fitness_values.append(base_fitness + penalty)
        
        es.tell(solutions, fitness_values)
        iteration += 1
        
        # Track progress
        best_fitness_history.append(es.result.fbest)
        
        if iteration % 20 == 0:
            print(f"  Iteration {iteration}: Best fitness = {es.result.fbest:.6f}")
            print(f"    Standard deviation: {es.sigma:.6f}")
    
    # Analyze results
    result = es.result
    print(f"\nOptimization results:")
    print(f"  Best solution: {format_array(result.xbest)}")
    print(f"  Best fitness: {result.fbest:.6f}")
    print(f"  Function evaluations: {result.evaluations}")
    print(f"  Convergence: {es.stop()}")
    
    # Check constraint satisfaction
    violations = 0
    for i, xi in enumerate(result.xbest):
        if xi < bounds[i][0] or xi > bounds[i][1]:
            violations += 1
    
    print(f"  Constraint violations: {violations}/{dimensions}")
    print(f"  Improvement: {best_fitness_history[0]:.6f} → {best_fitness_history[-1]:.6f}")
    
    print("✓ Multi-dimensional constrained optimization working")
    print("cma example 2 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    import traceback
    traceback.print_exc()
    raise
    
print("Example 2 finished")

## Example 3: Adaptive Parameter Control and Monitoring

In [ ]:
import cma
import numpy as np

# Example 3: Adaptive parameter control and optimization monitoring
print("Example 3: Testing adaptive parameter control")

# Helper function for formatting numpy arrays
def format_array(arr, precision=4):
    """Format numpy array for printing"""
    if len(arr) > 6:
        formatted = ', '.join(f'{x:.{precision}f}' for x in arr[:3])
        formatted += ', ..., '
        formatted += ', '.join(f'{x:.{precision}f}' for x in arr[-3:])
        return '[' + formatted + ']'
    return '[' + ', '.join(f'{x:.{precision}f}' for x in arr) + ']'

try:
    def noisy_function(x):
        """Function with noise to test robustness"""
        base_value = sum((xi - i * 0.5)**2 for i, xi in enumerate(x))
        noise = np.random.normal(0, 0.1)  # Add small amount of noise
        return base_value + noise
    
    def ill_conditioned_function(x):
        """Ill-conditioned quadratic function"""
        # Different scales for different dimensions
        scales = [1, 100, 10000] + [1] * (len(x) - 3)
        return sum(scale * (xi - 1)**2 for scale, xi in zip(scales, x))
    
    print("Testing adaptive parameters with ill-conditioned function:")
    
    # Problem setup
    dimensions = 6
    initial_guess = [0.0] * dimensions
    sigma0 = 2.0
    
    # Configure with adaptive settings
    options = {
        'maxiter': 150,
        'tolfun': 1e-10,
        'tolx': 1e-12,
        'verb_disp': 0,
        'CMA_stds': None,  # Use adaptive coordinate-wise standard deviations
    }
    
    es = cma.CMAEvolutionStrategy(initial_guess, sigma0, options)
    
    # Track adaptation
    sigma_history = []
    condition_history = []
    
    iteration = 0
    while not es.stop() and iteration < 120:
        solutions = es.ask()
        fitness_values = [ill_conditioned_function(x) for x in solutions]
        es.tell(solutions, fitness_values)
        
        # Monitor adaptation
        sigma_history.append(es.sigma)
        
        # Condition number of covariance matrix
        eigenvalues = es.D**2
        condition_number = max(eigenvalues) / min(eigenvalues) if min(eigenvalues) > 0 else float('inf')
        condition_history.append(condition_number)
        
        iteration += 1
        
        if iteration % 30 == 0:
            print(f"  Iteration {iteration}:")
            print(f"    Best fitness: {es.result.fbest:.6f}")
            print(f"    Step size (sigma): {es.sigma:.6f}")
            print(f"    Condition number: {condition_number:.2e}")
    
    # Final results and adaptation analysis
    result = es.result
    print(f"\nFinal optimization results:")
    print(f"  Best solution: {format_array(result.xbest)}")
    print(f"  Best fitness: {result.fbest:.8f}")
    print(f"  Function evaluations: {result.evaluations}")
    
    # Adaptation analysis
    print(f"\nAdaptation analysis:")
    print(f"  Initial sigma: {sigma_history[0]:.6f}")
    print(f"  Final sigma: {sigma_history[-1]:.6f}")
    print(f"  Sigma change ratio: {sigma_history[-1]/sigma_history[0]:.6f}")
    print(f"  Final condition number: {condition_history[-1]:.2e}")
    
    # Check coordinate-wise standard deviations
    coordinate_stds = es.sigma * es.D
    print(f"  Coordinate standard deviations: {format_array(coordinate_stds[:4])}")
    print(f"  Std deviation ratio (max/min): {max(coordinate_stds)/min(coordinate_stds):.2f}")
    
    print("✓ Adaptive parameter control working")
    print("cma example 3 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    import traceback
    traceback.print_exc()
    raise
    
print("Example 3 finished")

## Example 4: Multi-Modal Optimization and Restarts

In [ ]:
import cma
import numpy as np

# Example 4: Multi-modal optimization with restart strategies
print("Example 4: Testing multi-modal optimization with restarts")

# Helper function for formatting arrays
def format_array(arr, precision=4):
    """Format numpy array for printing"""
    arr = np.asarray(arr)
    if len(arr) > 6:
        formatted = ', '.join(f'{x:.{precision}f}' for x in arr[:3])
        formatted += ', ..., '
        formatted += ', '.join(f'{x:.{precision}f}' for x in arr[-3:])
        return '[' + formatted + ']'
    return '[' + ', '.join(f'{x:.{precision}f}' for x in arr) + ']'

try:
    def multi_modal_function(x):
        """Complex multi-modal function with multiple local optima"""
        # Rastrigin-like function with multiple local minima
        A = 10
        n = len(x)
        return A * n + sum(xi**2 - A * np.cos(2 * np.pi * xi) for xi in x)
    
    def himmelblau_function(x):
        """Himmelblau's function - has 4 global minima"""
        if len(x) < 2:
            return float('inf')
        return (x[0]**2 + x[1] - 11)**2 + (x[0] + x[1]**2 - 7)**2
    
    print("Testing multi-modal optimization with restarts:")
    
    # Multi-restart strategy
    dimensions = 4
    num_restarts = 3
    results = []
    
    for restart in range(num_restarts):
        print(f"\n  Restart {restart + 1}/{num_restarts}:")
        
        # Random initial guess for each restart
        initial_guess = np.random.uniform(-3, 3, dimensions)
        sigma0 = 1.5
        
        options = {
            'maxiter': 80,
            'tolfun': 1e-8,
            'seed': restart + 42,  # Different seed for each restart
            'verb_disp': 0
        }
        
        es = cma.CMAEvolutionStrategy(list(initial_guess), sigma0, options)
        
        # Optimization loop
        iteration = 0
        while not es.stop() and iteration < 60:
            solutions = es.ask()
            fitness_values = [multi_modal_function(x) for x in solutions]
            es.tell(solutions, fitness_values)
            iteration += 1
        
        # Store result
        result = es.result
        results.append({
            'restart': restart + 1,
            'initial_guess': initial_guess,
            'best_solution': result.xbest,
            'best_fitness': result.fbest,
            'evaluations': result.evaluations,
            'iterations': result.iterations
        })
        
        print(f"    Initial guess: {format_array(initial_guess[:3], 3)}")
        print(f"    Best fitness: {result.fbest:.6f}")
        print(f"    Function evaluations: {result.evaluations}")
    
    # Analyze results across restarts
    print(f"\nMulti-restart analysis:")
    
    best_overall = min(results, key=lambda r: r['best_fitness'])
    print(f"  Best overall fitness: {best_overall['best_fitness']:.6f} (restart {best_overall['restart']})")
    print(f"  Best solution: {format_array(best_overall['best_solution'])}")
    
    # Check diversity of solutions
    fitness_values = [r['best_fitness'] for r in results]
    fitness_std = np.std(fitness_values)
    print(f"  Fitness diversity (std): {fitness_std:.6f}")
    
    total_evaluations = sum(r['evaluations'] for r in results)
    print(f"  Total function evaluations: {total_evaluations}")
    
    # Test on 2D function to demonstrate multiple optima
    print(f"\n  Testing 2D multi-modal function (Himmelblau):")
    
    initial_2d = [0.0, 0.0]
    es_2d = cma.CMAEvolutionStrategy(initial_2d, 2.0, {'maxiter': 50, 'verb_disp': 0})
    
    iteration = 0
    while not es_2d.stop() and iteration < 40:
        solutions = es_2d.ask()
        fitness_values = [himmelblau_function(x) for x in solutions]
        es_2d.tell(solutions, fitness_values)
        iteration += 1
    
    result_2d = es_2d.result
    print(f"    2D optimization result: {format_array(result_2d.xbest)}")
    print(f"    2D best fitness: {result_2d.fbest:.6f}")
    
    print("✓ Multi-modal optimization with restarts working")
    print("cma example 4 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    import traceback
    traceback.print_exc()
    raise
    
print("Example 4 finished")

## Example 5: Production Optimization Pipeline

In [ ]:
import cma
import numpy as np
import time

# Example 5: Production-ready optimization pipeline with comprehensive features
print("Example 5: CMA-ES production optimization pipeline")

# Helper function for formatting arrays
def format_array(arr, precision=4):
    """Format numpy array for printing"""
    arr = np.asarray(arr)
    if len(arr) > 6:
        formatted = ', '.join(f'{x:.{precision}f}' for x in arr[:3])
        formatted += ', ..., '
        formatted += ', '.join(f'{x:.{precision}f}' for x in arr[-3:])
        return '[' + formatted + ']'
    return '[' + ', '.join(f'{x:.{precision}f}' for x in arr) + ']'

try:
    class OptimizationPipeline:
        def __init__(self, objective_function, dimensions, bounds=None):
            self.objective_function = objective_function
            self.dimensions = dimensions
            self.bounds = bounds
            self.optimization_history = []
            self.best_solutions = []
            
        def optimize(self, initial_guess=None, sigma0=1.0, max_evaluations=1000, 
                    tolerance=1e-8, random_restarts=1):
            """Run comprehensive optimization with monitoring and restarts"""
            
            start_time = time.time()
            all_results = []
            
            for restart in range(random_restarts):
                # Generate initial guess if not provided
                if initial_guess is None:
                    if self.bounds:
                        lower, upper = self.bounds
                        current_guess = np.random.uniform(lower, upper, self.dimensions)
                    else:
                        current_guess = np.random.uniform(-2, 2, self.dimensions)
                else:
                    current_guess = np.array(initial_guess) + 0.1 * np.random.randn(self.dimensions) * restart
                if self.bounds:
                    # cma requires x0 INSIDE the bounds (else: "argument of inverse must be within the given
                    # bounds"); the perturbed restart guess 0.2 + 0.1*N(0,1) left [0, 1] in ~11% of runs
                    current_guess = np.clip(current_guess, self.bounds[0], self.bounds[1])
                
                # Configure optimizer
                options = {
                    'maxfevals': max_evaluations // random_restarts,
                    'tolfun': tolerance,
                    'tolx': tolerance * 1e-2,
                    'verb_disp': 0,
                    'seed': restart * 123
                }
                
                if self.bounds:
                    options['bounds'] = [list(self.bounds[0]), list(self.bounds[1])]
                
                # Run optimization
                es = cma.CMAEvolutionStrategy(list(current_guess), sigma0, options)
                
                restart_history = []
                iteration = 0
                
                while not es.stop():
                    solutions = es.ask()
                    fitness_values = []
                    
                    for sol in solutions:
                        try:
                            fitness = self.objective_function(sol)
                            fitness_values.append(fitness)
                        except Exception as e:
                            fitness_values.append(float('inf'))  # Penalty for invalid solutions
                    
                    es.tell(solutions, fitness_values)
                    
                    # Record progress
                    restart_history.append({
                        'iteration': iteration,
                        'best_fitness': es.result.fbest,
                        'sigma': es.sigma,
                        'evaluations': es.result.evaluations
                    })
                    
                    iteration += 1
                
                result = es.result
                all_results.append({
                    'restart': restart,
                    'best_solution': result.xbest.copy(),
                    'best_fitness': result.fbest,
                    'evaluations': result.evaluations,
                    'iterations': result.iterations,
                    'history': restart_history,
                    'stop_condition': es.stop()
                })
            
            # Select best overall result
            best_result = min(all_results, key=lambda r: r['best_fitness'])
            
            # Store results
            optimization_run = {
                'timestamp': time.time(),
                'duration': time.time() - start_time,
                'best_result': best_result,
                'all_results': all_results,
                'total_evaluations': sum(r['evaluations'] for r in all_results)
            }
            
            self.optimization_history.append(optimization_run)
            self.best_solutions.append(best_result['best_solution'])
            
            return best_result
        
        def get_statistics(self):
            """Get comprehensive optimization statistics"""
            if not self.optimization_history:
                return None
                
            latest = self.optimization_history[-1]
            return {
                'runs_completed': len(self.optimization_history),
                'latest_best_fitness': latest['best_result']['best_fitness'],
                'latest_duration': latest['duration'],
                'total_evaluations': latest['total_evaluations'],
                'convergence_achieved': 'tolx' in latest['best_result']['stop_condition'] or 'tolfun' in latest['best_result']['stop_condition']
            }
    
    # Define test optimization problems
    def portfolio_optimization(weights):
        """Simplified portfolio optimization objective"""
        # Simulate returns and risk
        expected_returns = np.array([0.1, 0.12, 0.08, 0.15, 0.09])
        risk_matrix = np.random.rand(len(weights), len(weights))
        risk_matrix = risk_matrix @ risk_matrix.T  # Ensure positive semi-definite
        
        # Normalize weights
        weights = np.abs(weights)
        if np.sum(weights) > 0:
            weights = weights / np.sum(weights)
        
        # Risk-return tradeoff (minimize risk for given return)
        portfolio_return = np.dot(weights, expected_returns[:len(weights)])
        portfolio_risk = np.dot(weights, np.dot(risk_matrix[:len(weights), :len(weights)], weights))
        
        # Objective: minimize risk while maintaining reasonable return
        target_return = 0.1
        return_penalty = max(0, target_return - portfolio_return) * 100
        
        return portfolio_risk + return_penalty
    
    print("Running production optimization pipeline:")
    
    # Initialize pipeline
    dimensions = 5
    bounds = (np.zeros(dimensions), np.ones(dimensions))  # Portfolio weights between 0 and 1
    
    pipeline = OptimizationPipeline(
        objective_function=portfolio_optimization,
        dimensions=dimensions,
        bounds=bounds
    )
    
    # Run optimization
    result = pipeline.optimize(
        initial_guess=[0.2] * dimensions,
        sigma0=0.3,
        max_evaluations=800,
        tolerance=1e-6,
        random_restarts=2
    )
    
    # Display results
    print(f"\nOptimization Results:")
    print(f"  Best solution: {format_array(result['best_solution'])}")
    print(f"  Best fitness: {result['best_fitness']:.6f}")
    print(f"  Function evaluations: {result['evaluations']}")
    print(f"  Convergence: {result['stop_condition']}")
    
    # Get comprehensive statistics
    stats = pipeline.get_statistics()
    print(f"\nPipeline Statistics:")
    print(f"  Runs completed: {stats['runs_completed']}")
    print(f"  Duration: {stats['latest_duration']:.3f} seconds")
    print(f"  Total evaluations: {stats['total_evaluations']}")
    print(f"  Convergence achieved: {stats['convergence_achieved']}")
    
    # Validate solution
    weights = np.abs(result['best_solution'])
    normalized_weights = weights / np.sum(weights)
    print(f"  Normalized portfolio weights: {format_array(normalized_weights)}")
    print(f"  Weight sum: {np.sum(normalized_weights):.6f}")
    
    print(f"\nProduction pipeline capabilities:")
    capabilities = [
        "🎯 Multi-restart strategy for global optimization",
        "📊 Comprehensive progress tracking and statistics",
        "🛡️ Robust error handling for invalid solutions",
        "⚙️ Configurable convergence criteria and constraints",
        "🔄 Automated parameter adaptation and monitoring",
        "📈 Performance metrics and optimization history"
    ]
    
    for capability in capabilities:
        print(f"  {capability}")
    
    print("\n✓ Production optimization pipeline working")
    print("cma example 5 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    import traceback
    traceback.print_exc()
    raise
    
print("Example 5 finished")